# From data to geometry

This notebook takes a small table of numbers and turns each row into real
Blender geometry — a 3D bar chart that appears in the viewport as the cells
run.

Because the kernel lives *inside* Blender, there is no file to export and
re-import: every cell mutates the running scene directly. The last section
wires a slider to the bars so you can reshape the chart live.

In [ ]:
import os
import tempfile

import bpy
import numpy as np
import ipywidgets as widgets
from IPython.display import Image, display

print("Blender", bpy.app.version_string)

## Scene helpers

Two small helpers we reuse below: `fresh_scene()` clears everything and adds
a camera and sun so renders are lit, and `render()` shows the current frame
inline.

In [ ]:
COLLECTION = "DataBars"


def fresh_scene():
    """Empty the scene, then add a framed camera and a sun."""
    bpy.ops.object.select_all(action="SELECT")
    bpy.ops.object.delete()

    cam_data = bpy.data.cameras.new("Camera")
    cam = bpy.data.objects.new("Camera", cam_data)
    cam.location = (0.0, -22.0, 9.0)
    cam.rotation_euler = (np.radians(72), 0.0, 0.0)
    bpy.context.scene.collection.objects.link(cam)
    bpy.context.scene.camera = cam

    sun_data = bpy.data.lights.new("Sun", type="SUN")
    sun_data.energy = 4.0
    sun = bpy.data.objects.new("Sun", sun_data)
    sun.rotation_euler = (np.radians(35), np.radians(15), np.radians(20))
    bpy.context.scene.collection.objects.link(sun)


def render():
    """Render the active camera and display the result inline."""
    scene = bpy.context.scene
    scene.render.engine = "BLENDER_EEVEE_NEXT"
    scene.render.resolution_x = 640
    scene.render.resolution_y = 360
    bpy.ops.render.render()
    path = os.path.join(tempfile.gettempdir(), "jb_render.png")
    bpy.data.images["Render Result"].save_render(filepath=path)
    display(Image(filename=path))

## 1 · The data

A tiny inline dataset — average monthly rainfall in mm for one year. No
files, no network: just two NumPy arrays, so the notebook runs anywhere.

In [ ]:
months = np.array(
    ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
     "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
)
rainfall = np.array(
    [62, 47, 54, 41, 78, 99, 121, 110, 84, 67, 58, 70], dtype=float
)

# Quick text preview, no plotting library needed.
for name, value in zip(months, rainfall):
    bar = "#" * int(value / 4)
    print(f"{name}  {bar} {value:.0f} mm")

## 2 · One bar per row

Each value becomes a cube: positioned along X, scaled in Z by the rainfall.
The bars live in their own collection so we can find and rebuild them easily.

In [ ]:
fresh_scene()

# Start from an empty, dedicated collection.
if COLLECTION in bpy.data.collections:
    col = bpy.data.collections[COLLECTION]
    for obj in list(col.objects):
        bpy.data.objects.remove(obj, do_unlink=True)
else:
    col = bpy.data.collections.new(COLLECTION)
    bpy.context.scene.collection.children.link(col)

heights = rainfall / 20.0  # scale mm down to a pleasant size
bars = []
for i, h in enumerate(heights):
    bpy.ops.mesh.primitive_cube_add(size=1.0)
    bar = bpy.context.active_object
    bar.name = f"bar_{months[i]}"
    bar.scale = (0.4, 0.4, float(h))
    bar.location = (i - len(heights) / 2.0, 0.0, h / 2.0)
    for c in bar.users_collection:  # move it into our collection
        c.objects.unlink(bar)
    col.objects.link(bar)
    bars.append(bar)

render()

## 3 · Color by value

Taller bars run warm, shorter bars run cool. We give each bar a Principled
material and set its base color from the normalized value.

In [ ]:
def value_to_color(t):
    """Map t in [0, 1] to a blue -> red gradient."""
    return (t, 0.15, 1.0 - t, 1.0)


span = rainfall.max() - rainfall.min()
norm = (rainfall - rainfall.min()) / span
for bar, t in zip(bars, norm):
    mat = bpy.data.materials.new(name=f"mat_{bar.name}")
    mat.use_nodes = True
    bsdf = mat.node_tree.nodes["Principled BSDF"]
    bsdf.inputs["Base Color"].default_value = value_to_color(float(t))
    bar.data.materials.clear()
    bar.data.materials.append(mat)

render()

## 4 · Reshape it live

This is the part a static export can't do. The slider below scales every bar
in real time — drag it and watch the bars grow and shrink in Blender's
viewport, no re-render needed.

In [ ]:
exaggeration = widgets.FloatSlider(
    value=1.0, min=0.2, max=3.0, step=0.1, description="scale"
)
base_heights = heights.copy()


def reshape(_change=None):
    factor = exaggeration.value
    for bar, h in zip(bars, base_heights):
        new_h = float(h * factor)
        bar.scale.z = new_h
        bar.location.z = new_h / 2.0
    bpy.context.view_layer.update()


exaggeration.observe(reshape, names="value")
display(exaggeration)

That's the whole loop: **data → geometry → live edit**, without ever leaving
the notebook. Swap in your own arrays in section 1 and rerun section 2 to
chart anything you like.